# 02 - Features and simple baselines

Simple methods are tested before training a ranking model. This gives a clear result for later comparison.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from app.data import load_interactions, load_properties
from app.data.split import temporal_split
from app.evaluation.offline_evaluation import evaluate_user_rankings
from app.features.property_features import (
    amenity_overlap, availability_fit, budget_fit, build_listing_text, quality_fit,
)
from app.features.user_features import user_preferences_from_history
from app.recommendation.cold_start import cold_start_candidates
from app.retrieval.content_based import ContentRetriever

properties = load_properties()
interactions = load_interactions()
print(f'{len(properties):,} listings and {len(interactions):,} interactions loaded')

## Listing text

The text feature joins the title, description, area, property type and amenities. It is used by TF-IDF search.

In [ ]:
text_examples = properties.head(3).copy()
text_examples['listing_text'] = [build_listing_text(row) for row in text_examples.to_dict('records')]
display(text_examples[['property_id', 'title', 'listing_text']])

## Preference features

This test search makes each score easy to inspect. A score near 1 means a better fit.

In [ ]:
preferences = {
    'query': 'entire apartment in Berlin with Wifi and kitchen',
    'city': 'Berlin',
    'max_budget': 180,
    'bedrooms': 1,
    'property_types': ['Entire home/apt'],
    'amenities': ['Wifi', 'Kitchen', 'Washer'],
}

feature_frame = properties.copy()
feature_frame['budget_score'] = feature_frame['price'].map(
    lambda price: budget_fit(price, preferences['max_budget'])
)
feature_frame['amenity_score'] = feature_frame['amenities'].map(
    lambda values: amenity_overlap(values, preferences['amenities'])
)
feature_frame['quality_score'] = feature_frame['rating'].map(quality_fit)
feature_frame['availability_score'] = feature_frame['availability_365'].map(availability_fit)

score_columns = ['budget_score', 'amenity_score', 'quality_score', 'popularity_score', 'availability_score']
display(feature_frame[['property_id', 'title', 'price', *score_columns]].sort_values('budget_score', ascending=False).head(10).round(3))

In [ ]:
feature_frame.set_index('property_id')[score_columns].head(12).plot(
    kind='bar', figsize=(12, 5), ylim=(0, 1), title='Feature scores for the test search'
)
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Baseline 1: popular and well-rated listings

This baseline is useful when a new user has no history.

In [ ]:
cold_start = cold_start_candidates(properties, city='Berlin', top_k=10)
display(cold_start[['property_id', 'title', 'price', 'rating', 'popularity_score', 'availability_365']])

## Baseline 2: content search

TF-IDF gives more weight to words that help separate one listing from another. FAISS or NumPy then finds the closest vectors.

In [ ]:
content_model = ContentRetriever().fit(properties)
content_scores = content_model.search(preferences['query'], top_k=min(10, len(properties)))

content_results = properties[properties['property_id'].isin(content_scores)].copy()
content_results['semantic_score'] = content_results['property_id'].map(content_scores)
content_results = content_results.sort_values('semantic_score', ascending=False)
display(content_results[['property_id', 'title', 'neighborhood', 'semantic_score']].round(3))
print('Vector search engine:', content_model.index.engine)

## Preferences from user history

For an existing user, the system can estimate budget, bedrooms, property types and common amenities.

In [ ]:
most_active_user = interactions['user_id'].value_counts().index[0]
learned_preferences = user_preferences_from_history(properties, interactions, str(most_active_user))
user_items = interactions.loc[interactions['user_id'] == most_active_user, 'property_id']
user_history = properties[properties['property_id'].isin(user_items)]

print('User:', most_active_user)
print('Estimated preferences:', learned_preferences)
display(user_history[['property_id', 'title', 'price', 'property_type', 'amenities']])

## Offline check for the popularity baseline

The last interaction from each user is held out. This keeps later feedback out of the training period.

In [ ]:
train, test = temporal_split(interactions)
k = min(10, len(properties))
popular_ids = cold_start['property_id'].astype(str).tolist()
predictions = {}

for user_id in test['user_id'].astype(str).unique():
    seen = set(train.loc[train['user_id'].astype(str) == user_id, 'property_id'].astype(str))
    predictions[user_id] = [item for item in popular_ids if item not in seen][:k]

popularity_metrics = evaluate_user_rankings(predictions, test, k=k)
display(pd.Series(popularity_metrics, name='popularity_baseline').to_frame())

## Features kept for the next stage

- Popularity is a safe fallback, but it is not personal.
- Content search works for new listings because it only needs metadata.
- Budget and amenities should stay separate so their effect is easy to explain.
- The time-based baseline gives a minimum score that the hybrid ranker should beat.